## 加载module及输出数据源

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import pickle
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from matplotlib import pyplot as plt
import torch

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

## 数据加载

首先我们该想到的问题就是数据如何加载，一共有三种方式
1. 通过kaggle的Input传入
2. 通过一些成熟的module加载mnist
3. 从网络下载

## 定义全局参数

In [ ]:
class Config:
    def __init__(self):
        self.seed = 42 # 随机种子
        
        # 训练参数
        self.epochs = 200  # 轮数
        self.batch_size = 64  # 批次大小（
        self.val_ratio = 0.2  # 分割比例
        self.val_batch_size = 30  # 验证集批次大小
        self.learning_rate = 0.01  # 学习率
        self.layer_dims = [784, 50, 100, 10]  # 网络结构

        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.num_workers = 16 

        # 图像参数
        self.image_cols = None  # 调用update_from_dataset函数检测确定
        self.label_col = "label"  # 标签列名
        self.image_height = 28  # 图像高度
        self.image_width = 28  # 图像宽度
        self.num_channels = 1  # 通道数（1为灰度图，3为RGB）
        self.input_size = None  # 输入维度-调用下面函数计算
        self._calculate_input_size()  # 计算输入维度
        self.num_classes = 10  #输出分类

        #预处理
        self.normalize = True
        self.shuffle = True
        self.stratify  =True

        # 输入
        self.data_path = "/kaggle/input/"  # 数据根路径
        self.train_csv_path = os.path.join(self.data_path, "digit-recognizer/train.csv")  # 训练数据CSV路径
        self.test_csv_path = os.path.join(self.data_path, "digit-recognizer/test.csv")  # 测试数据CSV路径
        self.image_dir = os.path.join(self.data_path, "images/")  # 图像文件夹路径
        
        # 输出
        self.output_dir = "/kaggle/working/outputs/"  # 输出根目录
        self.model_save_path = os.path.join(self.output_dir, "best_model/")  # 模型保存路径
        self.submission_path = os.path.join(self.output_dir, "submission/")  # 提交文件路径
        

        os.makedirs(self.output_dir, exist_ok=True)

        

    def _calculate_input_size(self):
        self.input_size = self.image_height * self.image_width * self.num_channels  
        

    def update_from_dataset(self, df):
        # 检查标签列是否存在
        if self.label_col in df.columns:
            print(f"标签列: {self.label_col}")
        else:
            print("标签列不存在")

        # 自动检测图像特征列
        if self.image_cols is None:
            exclude_cols = [self.label_col, 'id', 'index']  # 排除非特征列
            self.image_cols = [col for col in df.columns if col not in exclude_cols]
            print(f"自动检测到 {len(self.image_cols)} 个图像特征列")

        # 确定类别数量
        if self.label_col in df.columns:
            self.num_classes = len(df[self.label_col].unique())
            print(f"类别数量: {self.num_classes}")
        else:
            print("标签列不存在，无法确定类别数量")

    def img_show(self, df, seed=42):
        """展示图片"""
        np.random.seed(seed)
        end = df.shape[0]
        index = np.random.randint(0, end)
        img = df.loc[index].to_numpy()[1:].reshape(28, -1)
        label = train_df[config.label_col][index]
        plt.xlabel(label)
        plt.imshow(img, cmap=plt.cm.binary_r)

    def print_info(self):
        print("=" * 50)
        print(f"设备: {self.device}")
        print(f"随机种子: {self.seed}")
        print(f"训练轮数: {self.epochs}")
        print(f"批次大小: 训练={self.batch_size}, 验证={self.val_batch_size}")
        print(f"学习率: {self.learning_rate}")
        print(f"图像尺寸: {self.image_height}x{self.image_width} (通道数: {self.num_channels})")
        print(f"输入维度: {self.input_size}")
        print(f"类别数量: {self.num_classes}")
        print(f"训练数据路径: {self.train_csv_path}")
        print(f"输出目录: {self.output_dir}")
        print("=" * 50)

## 定义工具类

In [ ]:
class Preprocess:
    def __init__(self, config):
        self.config = config
        self.start()

    @staticmethod
    def dataFrame2numpyArray(*args: pd.DataFrame):
        """将DataFrame转换为numpy数组"""
        res = [arg.to_numpy() for arg in args]
        return res[0] if len(res) == 1 else res
        
    def load_data(self, trainORtest='train'):
        """加载训练或测试数据"""
        print("[INFO] 加载训练或测试数据")
        if trainORtest == 'train':
            if not os.path.exists(self.config.train_csv_path):
                raise FileNotFoundError(f"训练文件数据不存在：{self.config.train_csv_path}")
            
            self.train_df = pd.read_csv(self.config.train_csv_path)
            print("="*50)
            print(f"原始训练数据形状：{self.train_df.shape}")
            
            self.config.update_from_dataset(self.train_df)
            print("="*50)
            
        elif trainORtest == 'test':
            if not os.path.exists(self.config.test_csv_path):
                raise FileNotFoundError(f"测试文件数据不存在：{self.config.test_csv_path}")
            
            self.test_df = pd.read_csv(self.config.test_csv_path)
            print("="*50)
            print(f"原始测试数据形状：{self.test_df.shape}")
                
            self.config.update_from_dataset(self.test_df)
            print("="*50)
    
    def split_data(self, val_ratio=0.2):
        """分离特征和标签并划分训练集/验证集"""
        print("[INFO] 分离特征和标签并划分训练集/验证集")
        if self.train_df is None:
            raise ValueError("请先加载训练数据")
            
        label_col = self.config.label_col
        print(f"标签列名: '{label_col}'")
        
        x_df = self.train_df.loc[:, self.train_df.columns != label_col]
        y_df = self.train_df[label_col]
        print(f"特征数据形状: {x_df.shape}, 标签数据形状: {y_df.shape}")

        x, y = self.dataFrame2numpyArray(x_df, y_df)

        print(f"验证集比例: {val_ratio*100}%")
        split_idx = int(len(x) * (1 - val_ratio))
        
        self.x_train, self.x_val = x[:split_idx], x[split_idx:]
        self.y_train, self.y_val = y[:split_idx], y[split_idx:]
        print(f"训练集特征: {self.x_train.shape}, 标签: {self.y_train.shape}")
        print(f"验证集特征: {self.x_val.shape}, 标签: {self.y_val.shape}")

    @staticmethod
    def normalization(*args):
        """数据归一化"""
        res = [arg / 255.0 for arg in args]
        return res[0] if len(res) == 1 else res

    def one_hot(self, num_classes=10):
        """将标签转换为one-hot编码"""
        print("[INFO] 将标签转换为one-hot编码")
        if self.y_train is None or self.y_val is None:
            raise ValueError("请先分割数据")
            
        self.y_train_onehot = np.zeros((len(self.y_train), num_classes))
        self.y_train_onehot[np.arange(len(self.y_train)), self.y_train] = 1
        
        self.y_val_onehot = np.zeros((len(self.y_val), num_classes))
        self.y_val_onehot[np.arange(len(self.y_val)), self.y_val] = 1

        print(f"训练集one-hot编码形状: {self.y_train_onehot.shape}")
        print(f"验证集one-hot编码形状: {self.y_val_onehot.shape}")

    def start(self):
        """执行完整的预处理流程"""
        self.load_data('train')
        self.split_data()
        
        if self.config.normalize:
            self.x_train_norm, self.x_val_norm = self.normalization(self.x_train, self.x_val)
        else:
            self.x_train_norm, self.x_val_norm = self.x_train, self.x_val

        self.one_hot()

    def get_data(self):
        return self.x_train_norm, self.x_val_norm, self.y_train_onehot, self.y_val_onehot

## 训练步骤实现

In [ ]:
class Model:
    def __init__(self, x_train_norm, x_val_norm, y_train_onehot, y_val_onehot, config):
        self.X_train = x_train_norm
        self.Y_train = y_train_onehot
        self.X_val = x_val_norm
        self.Y_val = y_val_onehot
        self.config = config
        
        # 训练记录
        self.train_losses = []
        self.val_accuracies = []
        self.parameters = None
        
        print("[INFO] 训练模块初始化完成")
        print(f"训练集形状: X_train={self.X_train.shape}, Y_train={self.Y_train.shape}")
        print(f"验证集形状: X_val={self.X_val.shape}, Y_val={self.Y_val.shape}\n")

    # 激活函数
    @staticmethod
    def sigmoid(x):
        return 1 / (1 + np.exp(-np.clip(x, -500, 500)))
        
    @staticmethod
    def softmax(x):
        exp_x = np.exp(x - np.max(x, axis=1, keepdims=True))  
        return exp_x / np.sum(exp_x, axis=1, keepdims=True)

    @staticmethod
    def relu(x):
        return np.maximum(0, x)

    @staticmethod
    def sigmoid_derivative(x):
        return x * (1 - x)

    @staticmethod
    def relu_derivative(x):
        return (x > 0).astype(np.float32)

    def initialize_weights(self):
        """初始化网络权重"""
        layer_dims = self.config.layer_dims
        print("[INFO] 初始化网络权重...")
        print(f"网络结构: {layer_dims}")
        
        if not isinstance(layer_dims, list) or len(layer_dims) < 2:
            raise ValueError("layer_dims must be a list with at least 2 elements")
        
        if not all(isinstance(dim, int) and dim > 0 for dim in layer_dims):
            raise ValueError("All elements in layer_dims must be positive integers")
            
        if self.config.seed is not None:
            np.random.seed(self.config.seed)
    
        self.parameters = {}
        for l in range(1, len(layer_dims)):
            self.parameters[f'W{l}'] = np.random.randn(layer_dims[l], layer_dims[l-1]) * 0.1
            self.parameters[f'b{l}'] = np.zeros((layer_dims[l], 1))
            print(f"初始化 W{l}: {self.parameters[f'W{l}'].shape}, b{l}: {self.parameters[f'b{l}'].shape}")
        
        print("[INFO] 权重初始化完成\n")

    def forward_propagation(self, X):
        """前向传播"""
        caches = {}
        A = X.T  # 转置以便与权重矩阵相乘
        caches['A0'] = A
        
        L = len(self.parameters) // 2
        
        # 隐藏层使用Relu激活
        for l in range(1, L):
            A_prev = A
            Z = np.dot(self.parameters[f'W{l}'], A_prev) + self.parameters[f'b{l}']
            A = self.relu(Z)
            caches[f'Z{l}'] = Z
            caches[f'A{l}'] = A
        
        # 输出层使用Softmax激活
        ZL = np.dot(self.parameters[f'W{L}'], A) + self.parameters[f'b{L}']
        AL = self.softmax(ZL.T).T  # 转置回来保持维度一致
        
        caches[f'Z{L}'] = ZL
        caches[f'A{L}'] = AL
        
        return AL, caches

    def compute_loss(self, AL, Y):
        """计算交叉熵损失"""
        m = Y.shape[0]
        AL = np.clip(AL, 1e-10, 1 - 1e-10)
        loss = -np.sum(Y * np.log(AL.T)) / m
        return loss

    def backward_propagation(self, AL, Y, caches):
        """反向传播"""
        grads = {}
        L = len(self.parameters) // 2
        m = Y.shape[0]
        
        # 输出层梯度
        dZL = (AL - Y.T) / m
        
        # 输出层的权重和偏置梯度
        grads[f'dW{L}'] = np.dot(dZL, caches[f'A{L-1}'].T)
        grads[f'db{L}'] = np.sum(dZL, axis=1, keepdims=True)
        
        # 隐藏层的梯度
        dA_prev = np.dot(self.parameters[f'W{L}'].T, dZL)
        
        for l in reversed(range(1, L)):
            # 当前层的梯度
            dZ = dA_prev * self.relu_derivative(caches[f'Z{l}'])
            
            # 计算权重和偏置的梯度
            grads[f'dW{l}'] = np.dot(dZ, caches[f'A{l-1}'].T)
            grads[f'db{l}'] = np.sum(dZ, axis=1, keepdims=True)
            
            # 计算前一层的梯度
            if l > 1:
                dA_prev = np.dot(self.parameters[f'W{l}'].T, dZ)
        
        return grads

    def update_parameters(self, grads, learning_rate):
        """更新参数"""
        L = len(self.parameters) // 2
        
        for l in range(1, L+1):
            self.parameters[f'W{l}'] -= learning_rate * grads[f'dW{l}']
            self.parameters[f'b{l}'] -= learning_rate * grads[f'db{l}']
        
        return self.parameters

    def predict(self, X):
        """预测"""
        AL, _ = self.forward_propagation(X)
        predictions = np.argmax(AL, axis=0)
        return predictions

    def compute_accuracy(self, X, Y):
        """计算准确率"""
        predictions = self.predict(X)
        labels = np.argmax(Y, axis=1)
        accuracy = np.mean(predictions == labels)
        return accuracy

    def __train_model(self):
        """训练模型内部实现"""
        self.initialize_weights()
        m = self.X_train.shape[0]
        
        print("[INFO] 开始模型训练...")
        print(f"[CONFIG] 训练参数: epochs={self.config.epochs}, lr={self.config.learning_rate}, batch_size={self.config.batch_size}")
        
        for epoch in range(self.config.epochs):
            # 随机打乱数据
            permutation = np.random.permutation(m)
            X_train_shuffled = self.X_train[permutation]
            Y_train_shuffled = self.Y_train[permutation]
            
            # 小批量训练
            for i in range(0, m, self.config.batch_size):
                X_batch = X_train_shuffled[i:i+self.config.batch_size]
                Y_batch = Y_train_shuffled[i:i+self.config.batch_size]
                
                # 前向传播
                AL, caches = self.forward_propagation(X_batch)
                
                # 计算损失
                loss = self.compute_loss(AL, Y_batch)
                
                # 反向传播
                grads = self.backward_propagation(AL, Y_batch, caches)
                
                # 更新参数
                self.update_parameters(grads, self.config.learning_rate)
            
            # 每个epoch结束后计算验证集准确率
            val_accuracy = self.compute_accuracy(self.X_val, self.Y_val)
            self.val_accuracies.append(val_accuracy)
            self.train_losses.append(loss)
            
            if epoch % 5 == 0 or epoch == self.config.epochs-1:
                print(f"[Accuracy] Epoch {epoch}/{self.config.epochs}, Loss: {loss:.4f}, Val Accuracy: {val_accuracy:.4f}")
                
        print("[INFO] 训练完成!")
        print(f"最终验证集准确率: {self.val_accuracies[-1]:.4f}")
        print("="*50)

    def visualize_training(self):
        """可视化训练过程中的损失和准确率"""
        plt.figure(figsize=(12, 5))
        
        # 绘制训练损失曲线
        plt.subplot(1, 2, 1)
        plt.plot(self.train_losses, label='Training Loss')
        plt.title('Training Loss')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.legend()
        
        # 绘制验证准确率曲线
        plt.subplot(1, 2, 2)
        plt.plot(self.val_accuracies, label='Validation Accuracy', color='orange')
        plt.title('Validation Accuracy')
        plt.xlabel('Epoch')
        plt.ylabel('Accuracy')
        plt.legend()
        
        plt.tight_layout()
        plt.show()

    def __save_weights(self):
        path = os.path.join(self.config.model_save_path, f'sample_weight_{self.val_accuracies[-1]}.pkl')
        if not os.path.exists(path):
            os.makedirs(path, exist_ok=True)
            print(f"保存路径已创建: {path}")
            
        # 将权重转换为可序列化的格式
        weights_dict = {}
        for key, value in self.parameters.items():
            weights_dict[key] = value
            
        # 保存为pickle文件
        with open(path, 'wb') as f:
            pickle.dump(weights_dict, f)
        
        print(f"权重已保存到 {path}") 

    def fit(self):
        """启动训练"""
        self.__train_model()
        self.visualize_training()
        # self.__save_weights()

## 运行

In [ ]:
config = Config()
config.print_info()

In [ ]:
preprocess = Preprocess(config)
x_train_norm, x_val_norm, y_train_onehot, y_val_onehot = preprocess.get_data()

In [ ]:
model = Model(x_train_norm, x_val_norm, y_train_onehot, y_val_onehot, config)
model.fit()

In [ ]:
preprocess.load_data(trainORtest='test')
x_test = preprocess.dataFrame2numpyArray(preprocess.test_df)

In [ ]:
img = x_test[2].reshape(28, -1)
plt.imshow(img, cmap=plt.cm.binary_r)

In [ ]:
preds = model.predict(x_test)

In [ ]:
submission = pd.DataFrame({
    "ImageId": np.arange(1, len(preds)+1),
    "Label": preds
})
submission_path = "/kaggle/working/submission.csv"
submission.to_csv(submission_path, index=False)
print("Saved:", submission_path)
submission.head()